<a href="https://colab.research.google.com/github/vitroid/research-scanner/blob/main/notebooks/research_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research scanner — notebook 版

`scripts/run_pipeline.py` をセル分割した版。Semantic Scholar から論文を引いてきて、
BERTopic でトピック化し、孤立論文を抽出して 2D / 3D の landscape を notebook に描画します。

## 想定環境

- **ローカル (このリポジトリ内)**: 直下の "Colab セットアップ" セルは clone・pip を何もしません。
  `.venv/bin/python -m jupyter lab notebooks/research_scan.ipynb` で開いてください。
- **Google Colab**: "Colab セットアップ" セルが GitHub から clone + `pip install -r requirements.txt`
  を実行します。`REPO_URL` を fork の URL に差し替えてください。
  pip が numpy / pandas などの ABI-sensitive なパッケージを差し替えた場合は、
  Colab ランタイムを自動で再起動します。再起動後は **同じセルをもう一度実行する** だけで
  途中から再開できます（clone と pip は冪等）。

`src/` 配下のモジュール (`fetch`, `topic_model`, `gap_analysis`, `visualize`) を再利用しているので、
パイプラインのロジックを書き換えたい時は `src/` を編集してから該当セルを再実行するだけで反映されます。


## 1. Colab セットアップ（ローカルでは自動的に skip）


In [ ]:
import os, sys, subprocess, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/vitroid/research-scanner.git"  # fork する場合は差し替え
REPO_DIR = "research-scanner"

IN_COLAB = "google.colab" in sys.modules


def _pkg_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


if IN_COLAB:
    if not Path(REPO_DIR).exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)

    # Snapshot ABI-sensitive packages BEFORE pip install; if pip ends up
    # swapping numpy/pandas/pyarrow the in-memory copies (already imported
    # by Colab's runtime) become incompatible with the new binaries and we
    # *must* restart the kernel for the new versions to take effect.
    sensitive = ("numpy", "pandas", "pyarrow", "scikit-learn")
    before = {p: _pkg_version(p) for p in sensitive}
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    after = {p: _pkg_version(p) for p in sensitive}
    swapped = {p: (before[p], after[p]) for p in sensitive if before[p] != after[p]}
    if swapped:
        print("These packages were upgraded/downgraded by pip:")
        for p, (b, a) in swapped.items():
            print(f"  {p}: {b} -> {a}")
        print(
            "Restarting Colab runtime so the new binaries are loaded.\n"
            "After it restarts, just rerun this cell — the clone + pip are\n"
            "idempotent so it'll move past quickly."
        )
        os.kill(os.getpid(), 9)  # Colab auto-restarts the kernel
    print("Colab setup done. cwd =", Path.cwd())
else:
    # ローカルで notebooks/ から実行した場合、リポジトリルートに登る
    here = Path.cwd().resolve()
    repo_root = here if (here / "src").exists() else here.parent
    os.chdir(repo_root)
    print("Local run, cwd =", Path.cwd())

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. 設定

Semantic Scholar の bulk search のクエリ構文（OR `|`、AND-default、フレーズ `"..."`、必須 `+`、除外 `-`、ワイルドカード `prefix*`、あいまい `word~N`）が使えます。


In [ ]:
# --- 検索条件 -----------------------------------------------------------------
QUERY = '("Bernal-Fowler" | "proton disorder"* | "hydrogen disorder"* | "Pauling entropy" ) -spin -monopole'
YEAR_FROM: int | None = 2000
YEAR_TO: int | None = None
MAX_PAPERS = 500

# --- トピックモデル -----------------------------------------------------------
EMBEDDING_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"  # 科学論文特化なら "allenai/specter2_base"
)
MIN_CLUSTER_SIZE = 10
CLUSTER_SELECTION_METHOD = (
    "leaf"  # "leaf" = 細かい多数クラスタ / "eom" = 少数の大クラスタ
)
LANDSCAPE_DIM = 3  # 2 or 3。3 にすると Scatter3d で回転可能になる

# --- 孤立判定 ------------------------------------------------------------------
ISOLATION_PERCENTILE = 95.0
EXCLUDE_KEYWORDS: list[str] = []  # 後段でも whole-word で除外したいキーワード

# --- 出力 ----------------------------------------------------------------------
RUN_NAME = "genice"  # outputs/<RUN_NAME>/ にすべて書き出す
OUTPUT_DIR = REPO_ROOT / "outputs" / RUN_NAME
# 既存キャッシュを再利用したい場合は run_pipeline.py と同じ "data/papers.parquet" を指す。
# クエリを切り替えるときは別名にするか、REFRESH=True で再取得。
CACHE_PATH = REPO_ROOT / "data" / "papers.parquet"
REFRESH = True  # True にすると Semantic Scholar から再取得

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Outputs ->", OUTPUT_DIR)
print("Cache   ->", CACHE_PATH, "(exists =", CACHE_PATH.exists(), ")")

## 3. 論文取得 (Semantic Scholar bulk search)

Semantic Scholar API キーを持っている場合は環境変数 `SEMANTIC_SCHOLAR_API_KEY` を設定するとレート制限が緩くなります。Colab では:

```python
import os; os.environ["SEMANTIC_SCHOLAR_API_KEY"] = "..."
```

をこのセルの上で実行。


In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s"
)

from src.fetch import (
    FetchConfig,
    fetch_papers,
    save_papers,
    load_papers,
    filter_with_abstract,
    exclude_keywords,
)

if CACHE_PATH.exists() and not REFRESH:
    papers_raw = load_papers(CACHE_PATH)
    print(f"Loaded cache: {len(papers_raw)} papers from {CACHE_PATH}")
else:
    cfg = FetchConfig(
        query=QUERY,
        year_from=YEAR_FROM,
        year_to=YEAR_TO,
        max_papers=MAX_PAPERS,
    )
    papers_raw = fetch_papers(cfg)
    save_papers(papers_raw, CACHE_PATH)
    print(f"Fetched {len(papers_raw)} papers and cached to {CACHE_PATH}")

papers = filter_with_abstract(papers_raw)
if EXCLUDE_KEYWORDS:
    papers = exclude_keywords(papers, EXCLUDE_KEYWORDS)
print(f"After filtering: {len(papers)} papers with informative abstracts")
papers.head()

## 4. トピックモデル + UMAP 投影

BERTopic は内部で sentence-transformers の埋め込み → UMAP → HDBSCAN を回します。Colab で **GPU ランタイム**にすると、最初の埋め込みステップが大きく速くなります。

`LANDSCAPE_DIM=3` の場合、可視化用の UMAP を 3 次元に投影します（クラスタリング用 UMAP の `umap_n_components=5` とは別）。


In [18]:
from src.topic_model import (
    TopicModelConfig,
    fit_topic_model,
    attach_topic_columns,
    save_topic_info,
)

tm_cfg = TopicModelConfig(
    embedding_model=EMBEDDING_MODEL,
    min_cluster_size=MIN_CLUSTER_SIZE,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    coords_dim=LANDSCAPE_DIM,
)
result = fit_topic_model(papers["abstract"].tolist(), cfg=tm_cfg)
papers_with_topics = attach_topic_columns(papers, result)

save_topic_info(result, OUTPUT_DIR / "topics.csv")
print(
    f"Topics found: {(result.topic_info['Topic'] >= 0).sum()}  "
    f"(outliers: {(papers_with_topics['topic'] == -1).sum()})"
)
result.topic_info.head(15)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Topics found: 3  (outliers: 44)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,44,-1_ice_hydrogen_proton_water,"[ice, hydrogen, proton, water, entropy, struct...",[The ability to form numerous crystalline modi...
1,0,27,0_ice_modes_vibrational_hydrogen,"[ice, modes, vibrational, hydrogen, vibrations...","[The hydrogen-disordered structure of ice, Ic,..."
2,1,23,1_ice_water_ih_simulations,"[ice, water, ih, simulations, phase, proton, q...",[We study the phase equilibrium between liquid...
3,2,20,2_ice_vi_ice vi_phase,"[ice, vi, ice vi, phase, hydrogen, transition,...",[Ice exhibits extraordinary structural variety...


## 5. 孤立論文の抽出

`topic == -1` (BERTopic の outlier) と、埋め込み空間での k-NN 平均距離が上位 `ISOLATION_PERCENTILE` 番目以上の論文を「孤立」とフラグ付けします。


In [19]:
from src.gap_analysis import GapConfig, isolated_papers, save_table

gap_cfg = GapConfig(isolation_percentile=ISOLATION_PERCENTILE)
isolated = isolated_papers(papers_with_topics, result.embeddings, gap_cfg)

save_table(isolated, OUTPUT_DIR / "papers_with_topics.csv")
save_table(isolated.loc[isolated["is_isolated"]], OUTPUT_DIR / "isolated_papers.csv")

print(f"Isolated: {int(isolated['is_isolated'].sum())} / {len(isolated)}")
isolated.loc[
    isolated["is_isolated"], ["title", "year", "topic_label", "isolation_score"]
].head(20)

Isolated: 45 / 114


,title,year,topic_label,isolation_score
0,Three peroxomorphic H2O2 adducts of antibiotic...,2024,Outlier,0.702484
1,Correction: Author Correction: Dynamics enhanc...,2018,Outlier,0.700307
2,Self-organisation of symbolic information,2015,Outlier,0.646848
3,A Strategy for the Analysis of the Far-Infrare...,2021,"ice, modes, vibrational, hydrogen",0.646726
4,Co-crystal or salt? A cautionary tale when inf...,2018,Outlier,0.561630
5,"Distinguishing between Clausius, Boltzmann and...",2019,Outlier,0.542888
7,Water on Pt(111): the importance of proton dis...,2007,Outlier,0.517415
9,Behavior and origin of hydrogen defects in nat...,2021,Outlier,0.502731
10,Modelling Water: A Lifetime Enigma.,2015,Outlier,0.497871
11,Hydrogen disorder in kaatialaite Fe[AsO2(OH)2]...,2021,Outlier,0.494611


## 6. Landscape の可視化

`save_figure` は self-contained な HTML を吐き、その中で plotly + サイドバー（凡例 + 概要パネル）を組み立てます。Notebook 内では同じ HTML を `show_landscape` ヘルパー経由で `<iframe srcdoc=...>` に埋め込むので、Colab / Cursor のサンドボックス内でも外部ファイルを参照せず動きます。


In [20]:
from src.visualize import landscape_figure, save_figure, show_landscape, build_report

fig = landscape_figure(isolated)
landscape_path = save_figure(fig, OUTPUT_DIR / "landscape.html")
report_path = build_report(
    query=QUERY,
    n_papers=len(papers),
    figure_path=landscape_path,
    topic_info=result.topic_info,
    isolated_df=isolated.loc[isolated["is_isolated"]],
    output_path=OUTPUT_DIR / "report.html",
)

print("Landscape ->", landscape_path)
print("Report    ->", report_path)

# Notebook 出力セル内に iframe srcdoc で landscape を直接埋め込む。
# 外部ファイルパスを参照しないので、Colab や Cursor のサンドボックス内でも動く。
show_landscape(fig)

Landscape -> /content/research-scanner/research-scanner/research-scanner/outputs/genice/landscape.html
Report    -> /content/research-scanner/research-scanner/research-scanner/outputs/genice/report.html


## 7. 個別論文を眺める

特定のトピックや個別の孤立論文を後追いするためのスニペット。`isolated` は `topic_label`, `isolation_score`, `is_isolated`, `umap_x/y/(z)` を含むデータフレームです。


In [21]:
import textwrap


def show_paper(row):
    print(f"[{row.get('year','?')}] {row['title']}")
    print(f"  topic       : {row.get('topic_label','?')}")
    print(f"  isolation   : {row.get('isolation_score', float('nan')):.3f}")
    if row.get("doi"):
        print(f"  doi         : https://doi.org/{row['doi']}")
    if isinstance(row.get("abstract"), str):
        print("  abstract    :")
        print(textwrap.indent(textwrap.fill(row["abstract"], width=88), "    "))
    print()


top_iso = isolated.loc[isolated["is_isolated"]].head(10)
for _, r in top_iso.iterrows():
    show_paper(r)

[2024] Three peroxomorphic H2O2 adducts of antibiotic furacin: the first cases of 2D hydrogen-bonded peroxide layers and concerted flip-flop hydrogen disorder of peroxide species.
  topic       : Outlier
  isolation   : 0.702
  doi         : https://doi.org/10.1039/d4ce00822g
  abstract    :
    Crystallization of antimicrobial compound furacin (nitrofurazone) from 96%, 50%, and 20%
    hydrogen peroxide (HP) led to three novel solvates C6H6N4O4•H2O2, C6H6N4O4•1.5(H2O2),
    and C6H6N4O4•3.5(H2O2), respectively. Surprisingly, more rich in hydrogen peroxide...

[2018] Correction: Author Correction: Dynamics enhanced by HCl doping triggers 60% Pauling entropy release at the ice XII–XIV transition
  topic       : Outlier
  isolation   : 0.700
  doi         : https://doi.org/10.1038/ncomms16189
  abstract    :
    Nature Communications 6: Article number: 7349 (2015); Published online 16 June 2015;
    Updated 20 June 2018 We became aware of a mistake related to the integration of peaks
   